# Hands-on Lab: SMS Spam Detection using Naïve Bayes (v2)

**Dataset:** SMS Spam Collection (UCI)  

---

### Overview

This lab builds a spam filter using the Naïve Bayes algorithm on the classic SMS Spam Collection dataset.
The pipeline covers text preprocessing (including stemming), feature extraction (BoW with bigrams and TF-IDF),
hyperparameter tuning via GridSearchCV, model comparison, evaluation under class imbalance, model persistence,
and a live classification function.

### Learning Objectives
- Understand why Naïve Bayes is effective for text classification
- Preprocess raw text: tokenisation, stop-word removal, stemming, vectorisation
- Recognise the difference between BoW (with bigrams) and TF-IDF representations
- Tune hyperparameters with GridSearchCV inside a sklearn Pipeline
- Evaluate a model correctly under class imbalance
- Compare two model variants (BoW vs TF-IDF) on the spam class F1-score
- Save and reload a trained model with joblib
- Deploy a live `classify_message()` function

---

## Background: Why Naïve Bayes Works for Spam

### Bayes' Theorem

Naïve Bayes classifiers are grounded in Bayes' Theorem:

```
P(Spam | Features) = ( P(Features | Spam) × P(Spam) ) / P(Features)
```

Where:
- **P(Spam | Features)** — posterior: probability a message is spam given its words
- **P(Features | Spam)** — likelihood: how probable these words are in spam
- **P(Spam)** — prior: base rate of spam in the dataset
- **P(Features)** — marginal: normalisation constant

### The "Naïve" Assumption

The model assumes each word's presence is **independent** of every other word given the class label. This is technically wrong ("free" and "prize" co-occur in spam more than chance), but it simplifies the likelihood calculation to a product of per-word probabilities and works surprisingly well in practice.

### Worked Numerical Example

Suppose an email has features F1 and F2, and our priors are:
- P(Spam) = 0.3, P(Not Spam) = 0.7
- P(F1|Spam) = 0.4, P(F2|Spam) = 0.5
- P(F1|Not Spam) = 0.2, P(F2|Not Spam) = 0.3

Under the naïve assumption:
```
P(F1,F2|Spam)     = 0.4 × 0.5 = 0.20
P(F1,F2|Not Spam) = 0.2 × 0.3 = 0.06

P(F1,F2) = (0.20 × 0.3) + (0.06 × 0.7) = 0.06 + 0.042 = 0.102

P(Spam|F1,F2)     = (0.20 × 0.3) / 0.102 ≈ 0.588
P(Not Spam|F1,F2) = (0.06 × 0.7) / 0.102 ≈ 0.412
```

Since 0.588 > 0.412, the email is classified as **spam**.

### MultinomialNB vs Other Variants

| Variant | Best for | Why |
|---|---|---|
| **MultinomialNB** | Word count / BoW features | Designed for discrete counts |
| BernoulliNB | Binary presence/absence features | Penalises absent words too |
| GaussianNB | Continuous features | Assumes Gaussian distribution |

For BoW and TF-IDF, **MultinomialNB** is the natural fit.

---

## Step 1: Import Required Packages

In [ ]:
import re
import joblib
import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

## Step 2: Load and Inspect the Dataset

In [ ]:
data = pd.read_csv('SMSSpamCollection.csv')
print(data.head())
print(f"\nDataset shape: {data.shape}")
print(f"\nLabel distribution:\n{data['Label'].value_counts()}")
print(f"\nMissing values:\n{data.isnull().sum()}")
print(f"\nDuplicate entries: {data.duplicated().sum()}")

# Remove duplicates if any
data = data.drop_duplicates()
print(f"\nShape after deduplication: {data.shape}")

## Step 3: Preprocess the Text

Preprocessing pipeline (in order):
1. **Lowercase** — treat "Free" and "free" as the same token
2. **Remove non-alpha characters** — strip numbers and punctuation (keeping `$` and `!` as spam signals)
3. **Tokenise** — split into individual words
4. **Remove stop words** — drop uninformative common words ("the", "and", "is")
5. **Stem** — reduce words to their root form (`running` → `run`, `prizes` → `prize`)
6. **Rejoin** — convert token list back to a string for vectorisers

**Why stemming matters here:** Without stemming, "win", "winner", and "winning" are counted as three distinct features. Stemming collapses them to one, cutting vocabulary size and improving generalisation — especially useful given the 5,500-message dataset size.

In [ ]:
data['Label'] = data['Label'].map({'spam': 1, 'ham': 0})

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    """Full preprocessing pipeline: lowercase → strip → tokenise → stop words → stem → rejoin."""
    # 1. Lowercase
    text = text.lower()
    # 2. Keep letters, whitespace, $ and ! (both are meaningful spam signals)
    text = re.sub(r'[^a-z\s$!]', '', text)
    # 3. Tokenise
    tokens = text.split()
    # 4. Remove stop words
    tokens = [w for w in tokens if w not in stop_words]
    # 5. Stem
    tokens = [stemmer.stem(w) for w in tokens]
    # 6. Rejoin
    return ' '.join(tokens)

data['cleaned_message'] = data['Message'].apply(preprocess_text)

print("=== BEFORE vs AFTER PREPROCESSING ===")
print(data[['Message', 'cleaned_message']].head(10))

## Step 4: Train/Test Split

**Critical rule:** split *before* vectorising. Fitting the vectoriser on the full dataset leaks test-set vocabulary into the feature space, artificially inflating reported performance. The correct pattern is:
- `fit_transform()` on training data only
- `transform()` (no fitting) on test data

With a Pipeline (Step 5), this is enforced automatically.

In [ ]:
X = data['cleaned_message']
y = data['Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set:  {X_test.shape[0]} samples")
print(f"\nSpam rate — train: {y_train.mean():.2%} | test: {y_test.mean():.2%}")

## Step 5: Feature Extraction Preview

### BoW with Bigrams

Using `ngram_range=(1,2)` captures both single words and two-word phrases. This matters for spam because **bigrams carry context that unigrams miss**:

| Unigram | Bigram |
|---|---|
| "free" (appears in ham too) | "free prize" → near-certain spam |
| "win" | "you win" → spam signal |
| "urgent" | "urgent reply" → spam signal |

`max_df=0.9` removes words that appear in over 90% of messages (too common to discriminate). `min_df=2` drops words that appear in fewer than 2 messages (too rare to generalise).

### TF-IDF

TF-IDF down-weights words frequent across all messages and up-weights words distinctive to specific messages. If "call" appears in 80% of all messages, its TF-IDF score is low even in a spam message heavy with "call" — reducing noise that BoW would amplify.

In [ ]:
# Quick preview only — actual vectorisation happens inside the Pipelines below
preview_vec = CountVectorizer(ngram_range=(1, 2), max_df=0.9, min_df=2)
preview_bow = preview_vec.fit_transform(X_train)

bow_df = pd.DataFrame(
    preview_bow.toarray(),
    columns=preview_vec.get_feature_names_out()
)
print(f"BoW feature matrix shape: {preview_bow.shape}")
print("\nBoW preview (first 5 rows, first 10 features):")
print(bow_df.iloc[:5, :10])

preview_tfidf = TfidfVectorizer(ngram_range=(1, 2), max_df=0.9, min_df=2)
preview_tfidf_mat = preview_tfidf.fit_transform(X_train)

tfidf_df = pd.DataFrame(
    preview_tfidf_mat.toarray(),
    columns=preview_tfidf.get_feature_names_out()
)
print(f"\nTF-IDF feature matrix shape: {preview_tfidf_mat.shape}")
print("\nTF-IDF preview (first 5 rows, first 10 features):")
print(tfidf_df.iloc[:5, :10])

## Step 6: Build Pipelines and Tune with GridSearchCV

A `Pipeline` chains vectorisation → classification into a single object. This guarantees:
- The vectoriser is always fitted on training data only (no leakage)
- The same transformation is applied consistently at prediction time
- GridSearchCV can search over both vectoriser and classifier parameters simultaneously

`GridSearchCV` with `cv=5` runs 5-fold cross-validation for each candidate `alpha` value. We optimise on **F1-score (spam class)** rather than accuracy — the right metric under class imbalance.

The `alpha` parameter is Laplace smoothing: it prevents zero probabilities for words unseen during training. Lower alpha = less smoothing = more sensitive to training data.

In [ ]:
# --- Pipeline 1: BoW with bigrams ---
bow_pipeline = Pipeline([
    ('vectorizer', CountVectorizer(ngram_range=(1, 2), max_df=0.9, min_df=2)),
    ('classifier', MultinomialNB())
])

param_grid = {'classifier__alpha': [0.01, 0.1, 0.15, 0.2, 0.5, 0.75, 1.0]}

bow_grid = GridSearchCV(bow_pipeline, param_grid, cv=5, scoring='f1', n_jobs=-1)
bow_grid.fit(X_train, y_train)

print(f"BoW best alpha: {bow_grid.best_params_}")
print(f"BoW best CV F1: {bow_grid.best_score_:.4f}")

In [ ]:
# --- Pipeline 2: TF-IDF with bigrams ---
tfidf_pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer(ngram_range=(1, 2), max_df=0.9, min_df=2)),
    ('classifier', MultinomialNB())
])

tfidf_grid = GridSearchCV(tfidf_pipeline, param_grid, cv=5, scoring='f1', n_jobs=-1)
tfidf_grid.fit(X_train, y_train)

print(f"TF-IDF best alpha: {tfidf_grid.best_params_}")
print(f"TF-IDF best CV F1: {tfidf_grid.best_score_:.4f}")

## Step 7: Evaluate and Compare Both Models

We evaluate on the held-out test set. **Do not use accuracy as the primary metric** — with ~87% ham and ~13% spam, a classifier that labels everything as ham achieves 87% accuracy while being completely useless as a spam filter.

The metrics that matter:
- **Precision (spam):** of all flagged messages, how many were genuinely spam? Low = legitimate mail gets blocked.
- **Recall (spam):** of all actual spam, how many were caught? Low = spam slips through.
- **F1 (spam):** harmonic mean of precision and recall. For a spam filter, **recall is typically prioritised** — missing spam is worse than an occasional false positive.

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    print(f"\n{'='*50}")
    print(f" {name}")
    print(f"{'='*50}")
    print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['ham', 'spam']))

evaluate_model('BoW + Bigrams (best alpha)', bow_grid.best_estimator_, X_test, y_test)
evaluate_model('TF-IDF + Bigrams (best alpha)', tfidf_grid.best_estimator_, X_test, y_test)

In [ ]:
from sklearn.metrics import f1_score

bow_f1   = f1_score(y_test, bow_grid.best_estimator_.predict(X_test))
tfidf_f1 = f1_score(y_test, tfidf_grid.best_estimator_.predict(X_test))

winner = 'BoW' if bow_f1 >= tfidf_f1 else 'TF-IDF'

print("\n=== Head-to-Head: Spam Class F1 ===")
print(f"BoW    F1: {bow_f1:.4f}")
print(f"TF-IDF F1: {tfidf_f1:.4f}")
print(f"Winner: {winner}")

best_model = bow_grid.best_estimator_ if winner == 'BoW' else tfidf_grid.best_estimator_
print(f"\nSelected '{winner}' model for deployment.")

## Step 8: Save and Reload the Best Model

`joblib` serialises the entire Pipeline — vectoriser vocabulary, fitted parameters, and classifier weights — into a single file. This means:
- No retraining needed at inference time
- The preprocessing pipeline travels with the model
- New messages need only to go through `preprocess_text()` before prediction

In [ ]:
model_path = 'spam_detection_model.joblib'
joblib.dump(best_model, model_path)
print(f"Model saved → {model_path}")

# Reload and verify
loaded_model = joblib.load(model_path)
print("Model reloaded successfully.")

# Sanity check — predictions should match
assert (loaded_model.predict(X_test) == best_model.predict(X_test)).all()
print("Sanity check passed — loaded model predictions match original.")

## Step 9: Live Classification Function

In [ ]:
def classify_message(message, model=loaded_model, show_prob=True):
    """Classify a single raw SMS message as Spam or Ham.
    
    Args:
        message: Raw SMS string
        model: Trained sklearn Pipeline (default: loaded_model)
        show_prob: If True, print spam probability alongside label
    """
    cleaned = preprocess_text(message)
    prediction = model.predict([cleaned])[0]
    label = 'Spam' if prediction == 1 else 'Ham'
    if show_prob:
        prob = model.predict_proba([cleaned])[0][1]
        return f"{label} (spam probability: {prob:.2%})"
    return label

# --- Test messages ---
test_cases = [
    # Known spam
    "Thanks for your subscription to Ringtone UK your mobile will be charged £5/month Please confirm by replying YES or NO.",
    "FREE entry in a weekly competition to win an iPad. Just text WIN to 80085 now!",
    "Congratulations! You've won a $1000 Walmart gift card. Go to http://bit.ly/1234 to claim now.",
    "Urgent! Your account has been compromised. Verify your details here: www.fakebank.com/verify",
    # Known ham
    "Hey, are we still meeting for lunch tomorrow?",
    "Reminder: Your appointment is scheduled for tomorrow at 10am.",
]

print("=== Live Classification ===")
for msg in test_cases:
    print(f"\nMessage : {msg[:80]}{'...' if len(msg) > 80 else ''}")
    print(f"Result  : {classify_message(msg)}")

---

## Personal Analysis

### Class Imbalance and Why Accuracy Is Misleading

The dataset contains approximately 4,827 ham messages and 747 spam messages — an ~87/13 split. A naive classifier that labels everything as ham achieves 87% accuracy without learning anything useful. This is why **precision, recall, and F1-score on the spam class** are the metrics that actually matter:

- **Precision (spam):** of all messages flagged as spam, how many actually were? Low precision means legitimate messages get blocked — a frustrating user experience.
- **Recall (spam):** of all actual spam messages, how many were caught? Low recall means spam slips through — the filter fails its primary job.
- The F1-score balances both. For a spam filter, **recall is typically prioritised** — missing spam is worse than the occasional false positive.

### Why Naïve Bayes Works Well for This Task

Despite its "naïve" independence assumption, MultinomialNB performs strongly on text classification because:
- Spam messages genuinely contain distinctive vocabulary (`free`, `win`, `prize`, `urgent`) at very different rates compared to ham
- The independence assumption, while technically wrong, rarely hurts in practice for bag-of-words features
- Training is extremely fast even on large vocabularies, making it practical for real-time filtering
- The `alpha` smoothing parameter prevents zero-probability crashes on unseen words at inference time

### BoW vs TF-IDF: What to Expect

Both representations performed comparably in this lab, which is typical for short-text classification tasks. The key trade-offs:

| | BoW (Count) | TF-IDF |
|---|---|---|
| **Strength** | Raw frequency is a strong spam signal | Down-weights common words across all messages |
| **Weakness** | High-frequency non-spam words inflate counts | May under-weight repeated spam keywords |
| **Best when** | Spam vocabulary is highly distinctive | Vocabulary overlaps more between spam and ham |

For SMS spam specifically, BoW often holds its own because spam vocabulary (`free`, `prize`, `win`) is so distinctive that raw counts are sufficient.

### Why Bigrams Help

Unigrams lose context. The word `free` appears legitimately in ham (`free tomorrow?`, `feel free`). The bigram `free prize` is near-certain spam. Adding `ngram_range=(1,2)` lets the model capture these patterns without needing word-order awareness — a practical middle ground between pure BoW and sequence models.

### The Data Leakage Issue

A common mistake in text classification pipelines is fitting the vectoriser on the full dataset before splitting. This leaks test-set vocabulary into the feature space, artificially inflating reported metrics. The correct pattern — enforced by `Pipeline` in this notebook — is: **split first, then fit the vectoriser on training data only**, and use `transform()` (not `fit_transform()`) on the test set. The Pipeline makes this the default behaviour, not something you have to remember manually.

### GridSearchCV and Choosing Alpha

Alpha controls Laplace smoothing. Too high and the model becomes over-smoothed, treating all words as equally probable — losing discriminating power. Too low and unseen words at inference time can cause numerical instability. Cross-validated grid search finds the sweet spot on training data without touching the test set, giving a more honest estimate of generalisation performance.